# Notebook 01 — Generate Distillation Data
**Member: A** | Generates D1 (all traces) and D2 (filtered traces) via Groq API.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import subprocess
from pathlib import Path
from google.colab import userdata

# Credentials from Colab Secrets — do not paste tokens into notebook cells
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

repo = Path('/content/SC4001-Group133')
repo_url = 'https://github.com/TIFFMUN/SC4001-Group133.git'

if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', repo_url, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)

os.chdir(repo)
%pip install -q -r requirements.txt
%pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sc4001-group133 (pyproject.toml) ... done


In [10]:
# Original cache setup
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

[setup_cache] Cache redirected to Drive: /content/drive/MyDrive/SC4001


In [11]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [12]:
# Separate second-round data from the original 100-example run
import json
import shutil
from pathlib import Path

project = Path('/content/SC4001-Group133')
drive_root = Path('/content/drive/MyDrive/SC4001')
original_data = drive_root / 'data'
updated_data = drive_root / 'updated_data'
updated_data.mkdir(parents=True, exist_ok=True)

# src/distillation uses relative data/... paths. Point those at the new Drive folder.
local_data = project / 'data'
if local_data.is_symlink():
    if local_data.resolve() not in (original_data.resolve(), updated_data.resolve()):
        raise RuntimeError(f'Unexpected data link: {local_data.resolve()}')
    if local_data.resolve() != updated_data.resolve():
        local_data.unlink()
elif local_data.exists():
    # Preserve any local files in the cloned repo rather than deleting them
    backup = project / 'data_repo_backup'
    if backup.exists():
        raise RuntimeError('Local data/ exists and backup name is already taken.')
    shutil.move(str(local_data), str(backup))

if not local_data.exists() and not local_data.is_symlink():
    local_data.symlink_to(updated_data, target_is_directory=True)

assert local_data.resolve() == updated_data.resolve()

# Seed new run with previously generated traces, once (saves Groq requests).
old_raw = original_data / 'traces/raw_traces.jsonl'
new_raw = updated_data / 'traces/raw_traces.jsonl'
new_raw.parent.mkdir(parents=True, exist_ok=True)

if not new_raw.exists() and old_raw.exists():
    shutil.copy2(old_raw, new_raw)
    print('Copied existing teacher traces from the original run.')
elif new_raw.exists():
    print('Found second-round raw traces; generation will resume from them.')
else:
    print('No original raw traces found; generation will start from zero.')

# Check for invalid/duplicate records before appending
questions = set()
if new_raw.exists():
    with new_raw.open(encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                raise ValueError(f'Blank JSONL line: {line_no}')
            rec = json.loads(line)
            if not all(isinstance(rec.get(k), str) and rec[k].strip()
                       for k in ('question', 'ground_truth', 'trace')):
                raise ValueError(f'Invalid trace record: line {line_no}')
            if rec['question'] in questions:
                raise ValueError(f'Duplicate question: line {line_no}')
            questions.add(rec['question'])

print('Data folder:', local_data.resolve())
print('Traces available:', len(questions))

Copied existing teacher traces from the original run.
Data folder: /content/drive/MyDrive/SC4001/updated_data
Traces available: 100


In [13]:
# Updated sample limit; use the same teacher settings as the original run
import yaml

TARGET_SAMPLES = 1000
config_path = Path('configs/distillation.yaml')
with config_path.open() as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['max_samples'] = TARGET_SAMPLES

# Confirm all configured data outputs resolve inside updated_data, not original data
for key, value in cfg['output'].items():
    if not isinstance(value, str):
        continue
    path = Path(value)
    resolved = path.resolve() if path.is_absolute() else (Path.cwd() / path).resolve()
    if not resolved.is_relative_to(updated_data.resolve()):
        raise RuntimeError(f'Output {key} would write outside updated_data: {resolved}')

with config_path.open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

# Record the settings used for this run in Drive
with (updated_data / 'distillation_updated.yaml').open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print('Target samples:', cfg['dataset']['max_samples'])
print('Teacher model:', cfg['teacher']['model'])
print('Raw traces path:', (Path.cwd() / cfg['output']['raw_traces']).resolve())
print('Output location:', updated_data)

Target samples: 1000
Teacher model: qwen/qwen3.8-27b
Raw traces path: /content/drive/MyDrive/SC4001/updated_data/traces/raw_traces.jsonl
Output location: /content/drive/MyDrive/SC4001/updated_data


In [14]:
import os
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
print('Groq API key loaded from Colab Secrets.')

Groq API key loaded from Colab Secrets.


In [15]:
# Generate the remaining teacher traces; existing questions are skipped automatically
from src.distillation.generate_traces import generate_traces
generate_traces()

Generating traces:  12%|█▏        | 118/1000 [06:01<45:01,  3.06s/it]  


KeyboardInterrupt: 

In [16]:
import importlib
import time
from groq import RateLimitError

gt = importlib.import_module(
    "src.distillation.generate_traces"
)

def call_teacher_with_retry(
    client, prompt, model, max_tokens, temperature
):
    for attempt in range(8):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=max_tokens,
                temperature=temperature
            )
            return response.choices[0].message.content

        except RateLimitError:
            wait = min(60, 3 * (2 ** attempt))
            print(
                f"Rate limit reached. "
                f"Retry {attempt + 1}/8 after {wait}s"
            )
            time.sleep(wait)

        except Exception as e:
            print(f"API error: {e}")
            return None

    print("Maximum retries reached for this question.")
    return None

gt.call_teacher = call_teacher_with_retry

print("Groq retry handling enabled.")

Groq retry handling enabled.


In [17]:
gt.generate_traces()

Generating traces:  13%|█▎        | 128/1000 [03:43<2:29:55, 10.32s/it]

Rate limit reached. Retry 1/8 after 3s


Generating traces:  14%|█▍        | 139/1000 [07:40<4:43:18, 19.74s/it]

Rate limit reached. Retry 1/8 after 3s


Generating traces:  16%|█▌        | 155/1000 [13:41<4:43:52, 20.16s/it]

Rate limit reached. Retry 1/8 after 3s


... (40 outputs left out)

Generating traces:  53%|█████▎    | 528/1000 [2:53:07<18:07:31, 138.24s/it]

Rate limit reached. Retry 1/8 after 3s
Rate limit reached. Retry 2/8 after 6s
Rate limit reached. Retry 3/8 after 12s
Rate limit reached. Retry 4/8 after 24s


Generating traces:  53%|█████▎    | 528/1000 [2:53:49<2:35:23, 19.75s/it]  


KeyboardInterrupt: 

In [18]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/SC4001/updated_data/traces/raw_traces.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print("Successfully saved:", len(records))
print("Unique questions:", len(set(r["question"] for r in records)))

Successfully saved: 522
Unique questions: 522


In [19]:
# Check generated traces before building datasets
import json
from pathlib import Path

raw_path = Path(
    "/content/drive/MyDrive/SC4001/updated_data/traces/raw_traces.jsonl"
)

with raw_path.open(encoding="utf-8") as f:
    raw = [json.loads(line) for line in f if line.strip()]

question_count = len({row["question"] for row in raw})

print("Generated records:", len(raw))
print("Unique questions:", question_count)

assert len(raw) == question_count, "Duplicate questions found!"
assert all(
    row.get("question") and row.get("ground_truth") and row.get("trace")
    for row in raw
), "Missing data found!"

# Ensure the project uses updated_data, not the original data
project_data = Path("/content/SC4001-Group133/data")
updated_data = Path("/content/drive/MyDrive/SC4001/updated_data")

assert project_data.resolve() == updated_data.resolve(), (
    "Project is not linked to updated_data!"
)

print("Dataset path:", project_data.resolve())
print("Dataset is ready for filtering!")

Generated records: 522
Unique questions: 522
Dataset path: /content/drive/MyDrive/SC4001/updated_data
Dataset is ready for filtering!


In [20]:
# Rebuild updated D1 and D2 from the 522 saved traces
import shutil
from pathlib import Path

updated_data = Path("/content/drive/MyDrive/SC4001/updated_data")
project_data = Path("/content/SC4001-Group133/data")

# Safety checks
assert project_data.resolve() == updated_data.resolve()
assert (updated_data / "traces/raw_traces.jsonl").is_file()

# Delete only previously generated UPDATED outputs
for relative_path in (
    "traces/filtered_traces.jsonl",
    "processed/all",
    "processed/filtered",
):
    target = updated_data / relative_path

    if target.is_dir():
        shutil.rmtree(target)
    elif target.is_file():
        target.unlink()

# Filter correct-answer traces (D2)
from src.distillation.filter_traces import filter_traces
filter_traces()

# Build training and validation datasets
from src.distillation.build_dataset import build_dataset

build_dataset(use_filtered=False)  # Updated D1
build_dataset(use_filtered=True)   # Updated D2

print("Updated D1 and D2 built successfully!")

Filtering traces: 100%|██████████| 522/522 [00:00<00:00, 12980.00it/s]


Saving the dataset (0/1 shards):   0%|          | 0/469 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/53 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/367 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/41 [00:00<?, ? examples/s]

Updated D1 and D2 built successfully!


In [21]:
# Final dataset summary
import json
from datasets import load_from_disk

raw_path = updated_data / "traces/raw_traces.jsonl"
filtered_path = updated_data / "traces/filtered_traces.jsonl"

with raw_path.open(encoding="utf-8") as f:
    raw = [json.loads(line) for line in f if line.strip()]

with filtered_path.open(encoding="utf-8") as f:
    filtered = [json.loads(line) for line in f if line.strip()]

all_ds = load_from_disk(str(updated_data / "processed/all"))
filtered_ds = load_from_disk(str(updated_data / "processed/filtered"))

# Verify counts
assert len(raw) == len({r["question"] for r in raw})
assert len(filtered) == len({r["question"] for r in filtered})
assert {r["question"] for r in filtered} <= {r["question"] for r in raw}

assert len(all_ds["train"]) + len(all_ds["validation"]) == len(raw)
assert len(filtered_ds["train"]) + len(filtered_ds["validation"]) == len(filtered)

print("Raw traces:", len(raw))
print("Filtered traces:", len(filtered))

print("Updated D1:",
      len(all_ds["train"]), "train /",
      len(all_ds["validation"]), "validation")

print("Updated D2:",
      len(filtered_ds["train"]), "train /",
      len(filtered_ds["validation"]), "validation")

print("\nNotebook 01 completed successfully!")

Raw traces: 522
Filtered traces: 408
Updated D1: 469 train / 53 validation
Updated D2: 367 train / 41 validation

Notebook 01 completed successfully!
